# Lecture 3b — Lepton Propagation with **PROPOSAL**

### Neutrino Interactions, Simulation and Event Generation · *N. Kamp*

This is part of a **3-notebook** set following the simulation pipeline:
```
  FLUX  ->  INTERACTION  ->  PROPAGATION  ->  LIGHT + DETECTOR
  [ SIREN tables + injection + weights ]  [ PROPOSAL ]  [ Prometheus ]
     part a  (this set: a=SIREN, b=PROPOSAL, c=Prometheus)
```

**This notebook (part b)** takes the $\tau$ made in 3a and follows its **race between flying and
decaying** — the decay-length distribution, the double-bang resolvability window, and how **PROPOSAL**
samples the real decay point with stochastic energy loss.

## Setup

Run this first. On Colab it clones the repo so `helpers` and `data/` exist, then puts `src/` on the
path. **Set `REPO_URL` to your repository.** If a compiled package fails to install, keep going —
every stage falls back to a cached/synthetic file.

In [ ]:
# === Setup: make the repo's helper code importable (the Colab fix) ===
# Colab's GitHub browser loads ONLY this .ipynb -- the surrounding repo
# (src/helpers.py, data/) is NOT cloned. So we clone it into the runtime here.
import os, sys, subprocess

REPO_URL = 'https://github.com/USER/REPO.git'   # <-- set to your repo
SUBDIR   = 'Lecture3_Simulation'                # folder that holds src/ and data/

if 'google.colab' in sys.modules and not os.path.isdir('REPO'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, 'REPO'], check=True)

# Find the folder containing src/helpers.py, whether cloned (Colab) or in-repo (local):
_cands = ['REPO/' + SUBDIR, SUBDIR, '.', '..', '../..']
_base  = next((p for p in _cands if os.path.isdir(os.path.join(p, 'src'))), None)
assert _base, 'Could not find src/ -- check REPO_URL / SUBDIR.'
sys.path.insert(0, os.path.abspath(os.path.join(_base, 'src')))

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)
if 'google.colab' in sys.modules:
    pip('pyarrow', 'awkward')        # light deps for reading cached data
    # proposal (optional) compiles from source (~minutes); not needed live -- the
    # decay study below is exact and pure-numpy. Uncomment to build it yourself:
    # pip('proposal')

import numpy as np, matplotlib.pyplot as plt
import helpers as H
print('helpers loaded from:', H.__file__)

> **Running on Colab.** The disk is wiped each session, so re-run Setup every time. The compiled
> tools (SIREN, PROPOSAL, Prometheus) build from source on Colab — so **we don't build them live**;
> each stage falls back to a small cached/synthetic file. For a live SIREN injection, install a
> **prebuilt wheel** (build once, host it, `pip install <url>` — see the README).

## 1. Propagation — the tau's race between flying and decaying

A $\nu_\tau$ CC event makes a $\tau$. Whether you see a **double bang** depends on the tau
**decay length** $L = \gamma c\tau \approx 50\,\mathrm{m} \times (E_\tau/\mathrm{PeV})$ being long enough to
separate the two cascades, but short enough to decay inside the detector. **PROPOSAL** handles the
charged-lepton propagation and stochastic losses (Lecture 2). The decay length isn't fixed — it's
**exponentially distributed** — so below we sample it and turn it into a *double-bang efficiency*.

In [ ]:
# Mean (lab-frame) tau decay length vs energy: L = (E/m_tau) * c*tau.
E_tau = np.logspace(4, 7, 100)                 # 10 TeV - 10 PeV
L_mean = H.tau_mean_decay_length_m(E_tau)

plt.figure(figsize=(7,5))
plt.loglog(E_tau, L_mean, label='mean decay length')
plt.axhspan(10, 1000, alpha=.12, color='C2', label='resolvable window')
plt.axhline(125, ls='--', color='grey'); plt.text(1.2e4, 140, 'IceCube string spacing ~125 m')
plt.xlabel(r'$E_\tau$ [GeV]'); plt.ylabel('tau decay length [m]')
plt.legend(); plt.title('Why double-bangs live around a PeV'); plt.grid(alpha=.3, which='both'); plt.show()

In [ ]:
# ---- (illustrative) sampling the REAL decay point with PROPOSAL ----
# PROPOSAL also applies stochastic energy loss before the tau decays. Watch its
# units: energy in MeV, length in cm. It compiles from source, so not run live.
if H.have('proposal'):
    import proposal as pp
    prop = pp.Propagator(
        particle_def=pp.particle.TauMinusDef(),
        path_to_config_file='config_ice.json')      # PROPOSAL ships example configs
    state = pp.particle.ParticleState()
    state.position = pp.Cartesian3D(0, 0, 0)         # cm
    state.direction = pp.Cartesian3D(0, 0, 1)
    state.energy = 1e9                               # MeV  (= 1 PeV)
    track = prop.propagate(state, max_distance=1e5)  # cm
    print(f'PROPOSAL tau decayed after {track.final_state().propagated_distance/100:.1f} m')
else:
    print('proposal not installed -- using the exact exponential decay MC below.')

In [ ]:
# ---- decay-length distribution + double-bang efficiency (no PROPOSAL needed) ----
rng = np.random.default_rng(0)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# left: sampled decay-length distributions for a few tau energies
for E0 in (1e5, 1e6, 1e7):
    d = H.sample_tau_decay_length_m(E0, size=20000, rng=rng)
    ax[0].hist(d, bins=np.logspace(-1, 4, 60), histtype='step', density=True,
               label=f'{E0/1e6:g} PeV')
ax[0].axvspan(10, 1000, alpha=.12, color='C2')
ax[0].set_xscale('log'); ax[0].set_xlabel('decay length [m]'); ax[0].set_ylabel('pdf')
ax[0].legend(title=r'$E_\tau$'); ax[0].set_title('Decay length is exponential -> long tail')

# right: fraction decaying inside the resolvable window vs energy
Eg = np.logspace(4, 7.5, 200)
for (dmin, dmax, c) in [(10, 1000, 'C0'), (30, 1000, 'C1')]:
    ax[1].semilogx(Eg, H.double_bang_efficiency(Eg, dmin, dmax), color=c,
                   label=f'window {dmin}-{dmax} m')
ax[1].set_xlabel(r'$E_\tau$ [GeV]'); ax[1].set_ylabel('double-bang fraction')
ax[1].legend(); ax[1].set_title('Resolvable double-bang efficiency vs energy')
plt.tight_layout(); plt.show()

Epeak = Eg[np.argmax(H.double_bang_efficiency(Eg, 10, 1000))]
print(f'efficiency (10-1000 m window) peaks near {Epeak/1e6:.1f} PeV')

> **🔧 Try it.** The right panel is a *resolvability* curve: too low in energy and the tau decays
> within metres (one merged cascade); too high and it usually exits before decaying (a track). Raise
> the window's lower edge 10 m → 30 m (worse vertex resolution): what happens to the peak height and
> the energy where it peaks? Then fold in the tau-energy spectrum from **3a** to get the *physical* rate.

> **Note.** The efficiency above uses *pure exponential decay* (exact decay kinematics, no install
> needed). PROPOSAL's added value is the **stochastic energy loss before decay**: at the highest
> energies the tau sheds energy and doesn't fly the full $\gamma c\tau$, so the real high-energy tail
> falls a bit faster. The same `pp.Propagator` machinery handles **muons** (`MuMinusDef`) — a natural
> exercise: swap the particle and plot muon range vs energy (MIP + stochastic losses, Lecture 2).

## Wrap-up

**References:** Prometheus (arXiv:2304.14526), SIREN / LeptonInjector (arXiv:2012.10449),
PROPOSAL (Comput.Phys.Commun. 2019), nuSQuIDS (arXiv:2112.13804), SIREN flux tables
(Zenodo 20129082), Formaggio & Zeller (Rev.Mod.Phys. 2012).

**Go further:**
- Replace `TauMinusDef` with `MuMinusDef` and study the muon range / energy loss (Bethe-Bloch + stochastic).
- Use the sampled decay lengths to place the second cascade, then view it in **3c (Prometheus)**.
- Weight the efficiency curve by the 3a tau spectrum for a physical double-bang rate.